# Post-flight analysis

Reads one recorded flight (a `concopt inflight --record` CSV, the C2 schema)
against its own pre-flight plan (a `concopt report --out` CSV) and answers
five questions, in order: predicted vs actual overall, where the prediction
drifted over the flight, how the three weather sources agree with each
other, what the advisor's advice was worth, and whether the arrival's
assumed constants (`arrival.APPROACH_NM`/`APPROACH_MIN`/`APPROACH_FUEL_T`)
hold up.

This is a working analysis, run by hand after a flight -- not a dashboard.
Every cell below reuses concopt's own functions; nothing here reimplements
the model or the recorder.

**No real flight exists yet.** Section 1 builds a synthetic recording and
its matching report from the model itself (`concopt.replay`, Phase C3) so
every cell below runs end to end without one -- a synthetic flight agrees
with the model by construction, so Q1 and Q5 below should come out close to
exact; anything else means the harness or this notebook is wrong, not the
model. To analyze a REAL flight instead, skip section 1 and point
`RECORDING_CSV_PATH`/`REPORT_CSV_PATH` (next cell) at your own
`--record`/`--out` files.

In [ ]:
import contextlib
import datetime as dt
import io
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from concopt import arrival, fuel, limits
from concopt.atmos import isa, pressure_to_fl
from concopt.inflight import RECORD_SCHEMA_VERSION, run_inflight
from concopt.replay import build_synthetic_flight, replay_sources
from concopt.report import run_report
from concopt.route import build_legs, climb_cruise_segment, parse_pln
from concopt.search import local_to_departure_utc, resolve_tow_and_arrival

# Same colour for the same series in every cell below (notebooks/day-search-results.ipynb's convention).
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]  # blue, orange, green, red


## 1. Data source

`GENERATE_SYNTHETIC = True` builds a synthetic recording+report pair from
the model itself (`concopt.replay.build_synthetic_flight`, the same
`search.resolve_tow_and_arrival` call `concopt report` makes, flown back
through `run_inflight` via THE SEAM) into a scratch directory. Set it to
`False` and fill in the two paths below to analyze a real flight instead --
nothing past section 2 cares which one it's looking at.

In [ ]:
GENERATE_SYNTHETIC = True

PLN_PATH = "../tests/data/KJFKEGLL_CONC_01.pln"
ZFW_T = 92.0
LOCAL_DATE, LOCAL_HOUR = dt.date(2016, 2, 12), 10
REPLAY_SPEED = 900.0  # a ~3.2 h flight replays in well under a minute
# Must match the --cruise-mach the real flight's `concopt report`/`concopt
# inflight` runs used (2.00 default; 2.04 tries Mmo, see limits.py) -- every
# call below that takes cruise_mach is otherwise silently defaulting to
# limits.CRUISE_MACH regardless of what was actually flown, comparing the
# real flight against a predicted plan built for the wrong cruise speed.
CRUISE_MACH = limits.CRUISE_MACH

# Only used when GENERATE_SYNTHETIC is False -- point these at a real
# `concopt inflight --record` CSV and a real `concopt report --out` CSV.
RECORDING_CSV_PATH = "../data/inflight/recording.csv"
REPORT_CSV_PATH = "../data/report.csv"


In [ ]:
def _still_air_data(levels_hpa, legs):
    '''era5.load_legs_npz-shaped dict, ISA+0 and zero wind everywhere --
    same construction as tests/test_replay.py's own fixture. Real callers
    would pass real era5.reduce_to_legs .npz's here instead; the model
    doesn't care which.'''
    fl_at_level = pressure_to_fl(levels_hpa * 100.0)
    temp_at_level, _ = isa(fl_at_level * 30.48)
    n_time = 2
    n_legs = len(legs)
    times = np.array(["2016-01-01T00:00:00", "2026-12-31T00:00:00"], dtype="datetime64[ns]")
    u = np.zeros((n_time, len(levels_hpa), n_legs))
    v = np.zeros((n_time, len(levels_hpa), n_legs))
    t = np.broadcast_to(temp_at_level[None, :, None], (n_time, len(levels_hpa), n_legs)).copy()
    return dict(time=times, level=levels_hpa, u=u, v=v, t=t,
                cum_nm=np.array([leg.cum_nm for leg in legs]),
                track_deg=np.array([leg.track_deg for leg in legs]))


if GENERATE_SYNTHETIC:
    scratch = Path(tempfile.mkdtemp(prefix="concopt_postflight_"))

    plan = parse_pln(PLN_PATH)
    legs = build_legs(plan["waypoints"])
    mask = climb_cruise_segment(legs)
    cc_idx = np.flatnonzero(mask)
    cc_legs = [legs[i] for i in cc_idx]
    arrival_legs = [leg for leg, m in zip(legs, mask) if not m]
    arrival_nm = legs[-1].cum_nm - cc_legs[-1].cum_nm

    data = _still_air_data(np.array([150.0, 125.0, 100.0, 70.0]), legs)
    subsonic_data = _still_air_data(np.array([175.0, 200.0, 225.0, 250.0, 300.0, 400.0, 500.0]), arrival_legs)
    arrival_upper_data = _still_air_data(np.array([70.0, 100.0, 125.0, 150.0]), arrival_legs)
    for name, d in [("route_legs", data), ("route_legs_subsonic", subsonic_data),
                    ("route_legs_arrival_upper", arrival_upper_data)]:
        np.savez(scratch / f"{name}.npz", **d)

    # Still-air surface wind for the runway screen (concopt report --surface-npz).
    surface = {}
    for airport in ("KJFK", "EGLL"):
        surface[f"{airport}_time"] = data["time"]
        for var in ("u10", "v10", "i10fg"):
            surface[f"{airport}_{var}"] = np.zeros(len(data["time"]))
    np.savez(scratch / "surface.npz", **surface)

    departure_utc_ts = pd.Timestamp(local_to_departure_utc(LOCAL_DATE, LOCAL_HOUR))
    dep_i8 = np.array([departure_utc_ts.value], dtype="int64")

    RECORDING_CSV_PATH = scratch / "recording.csv"
    REPORT_CSV_PATH = scratch / "report.csv"
    print(f"synthetic scratch dir: {scratch}")


### 1a. The predicted plan

`concopt report` PRINTS the fuel plan and the arrival's decel/level/descent
split but only persists the waypoint table (time + `arrival_s`) to its own
CSV -- there is no fuel or per-segment breakdown in `report.csv` to read
back. Question 1 needs that breakdown, so this cell recomputes it directly:
the SAME `search.resolve_tow_and_arrival` call `concopt report` itself
makes (not a different model), captured before it's thrown away by
`print()`. `report.csv` is still written below (1b) and used as-is for
question 3's ERA5 series and as a cross-check on the arrival total.

Analyzing a REAL flight (`GENERATE_SYNTHETIC = False`): question 1 still
needs `predicted`/`predicted_cum_bounds` from this cell, so rerun it with
your real `era5.load_legs_npz` data/subsonic_data/arrival_upper_data (and
real `dep_i8`) in place of the still-air fixture below -- same call, real
weather.

In [ ]:
if GENERATE_SYNTHETIC:
    tow_arr, n_iterations, plan_flags, legs_out, weight_per_leg, climb, arrival_out = (
        resolve_tow_and_arrival(
            cc_legs, cc_idx, arrival_legs, arrival_nm, data, subsonic_data, dep_i8,
            zfw_t=ZFW_T, min_landing_fuel_t=fuel.MIN_LANDING_FUEL_T,
            arrival_upper_data=arrival_upper_data, cruise_mach=CRUISE_MACH,
        )
    )
    fuel_plan = fuel.fuel_plan(climb, legs_out, arrival_out, zfw_t=np.array([ZFW_T]),
                                min_landing_fuel_t=fuel.MIN_LANDING_FUEL_T,
                                tow_t=tow_arr, n_iterations=n_iterations, flags=plan_flags)

    climb_s = float(climb["time_min"][0]) * 60.0
    cruise_s = float(legs_out["accumulated_s"][0]) - climb_s
    a = {k: float(v[0]) for k, v in arrival_out.items()
         if k not in ("by_schedule", "flags") and np.ndim(v) == 1}

    predicted = pd.DataFrame([
        dict(segment="climb", time_s=climb_s, fuel_t=float(fuel_plan["climb_fuel_t"][0])),
        dict(segment="cruise", time_s=cruise_s, fuel_t=float(fuel_plan["cruise_fuel_t"][0])),
        dict(segment="decel", time_s=a["decel_time_min"] * 60.0, fuel_t=a["decel_fuel_t"]),
        dict(segment="level", time_s=a["level_time_min"] * 60.0, fuel_t=a["level_fuel_t"]),
        dict(segment="descent", time_s=a["descent_time_min"] * 60.0, fuel_t=a["descent_fuel_t"]),
        dict(segment="approach", time_s=arrival.APPROACH_MIN * 60.0, fuel_t=arrival.APPROACH_FUEL_T),
    ]).set_index("segment")

    # cum_nm each predicted segment ENDS at -- question 1 cuts the actual
    # flight at these same route positions (see that cell for why).
    decel_cum_nm = float(cc_legs[-1].cum_nm)
    predicted_cum_bounds = [
        0.0, float(climb["ground_dist_nm"][0]), decel_cum_nm,
        decel_cum_nm + a["decel_nm"],
        decel_cum_nm + a["decel_nm"] + a["level_nm"],
        decel_cum_nm + a["decel_nm"] + a["level_nm"] + a["descent_nm"],
        float(legs[-1].cum_nm),
    ]
    print(predicted)
    print(f"\npredicted total: {predicted['time_s'].sum():,.0f} s, "
          f"{predicted['fuel_t'].sum():.2f} t (fuel_plan trip_fuel_t: "
          f"{float(fuel_plan['trip_fuel_t'][0]):.2f} t)")


### 1b. Write `report.csv` and the synthetic recording

`run_report` writes exactly the CSV a real `concopt report --out` run
would. The recording comes from flying that same predicted profile back
through `run_inflight` with no SimConnect/Active Sky at all (THE SEAM,
Phase C3) -- `build_synthetic_flight` adds a little noise so this replay
is not reading the control points back verbatim (see its own docstring).
Per-tick advisor output is suppressed (~200 ticks of it); only the
recorder's own two state-machine notes print.

In [ ]:
if GENERATE_SYNTHETIC:
    with contextlib.redirect_stdout(io.StringIO()):
        run_report(PLN_PATH, scratch / "route_legs.npz", LOCAL_DATE, LOCAL_HOUR,
                    out_path=REPORT_CSV_PATH, zfw_t=ZFW_T,
                    min_landing_fuel_t=fuel.MIN_LANDING_FUEL_T,
                    subsonic_npz_path=scratch / "route_legs_subsonic.npz",
                    arrival_upper_npz_path=scratch / "route_legs_arrival_upper.npz",
                    surface_npz_path=scratch / "surface.npz",
                    cruise_mach=CRUISE_MACH)

    profile_df = build_synthetic_flight(
        PLN_PATH, data, dep_i8, subsonic_data=subsonic_data,
        arrival_upper_data=arrival_upper_data, zfw_t=ZFW_T,
        sample_interval_s=15.0, seed=1, cruise_mach=CRUISE_MACH,
    )
    state_source, weather_source = replay_sources(profile_df, replay_speed=REPLAY_SPEED)
    with contextlib.redirect_stdout(io.StringIO()) as ticks:
        run_inflight(PLN_PATH, interval_s=60.0, record_path=str(RECORDING_CSV_PATH),
                     state_source=state_source, weather_source=weather_source,
                     replay_speed=REPLAY_SPEED, live=False, cruise_mach=CRUISE_MACH)
    for line in ticks.getvalue().splitlines():
        if "detected" in line:
            print(line)


## 2. Load and check the schema

Fails clearly on an unrecognised `schema_version` rather than silently
reading a recording built by an older/newer recorder and producing nonsense
from columns that have since moved or disappeared.

In [ ]:
recording = pd.read_csv(RECORDING_CSV_PATH)
report_df = pd.read_csv(REPORT_CSV_PATH)

versions = recording["schema_version"].unique()
if len(versions) != 1 or int(versions[0]) != RECORD_SCHEMA_VERSION:
    raise ValueError(
        f"recording schema_version(s) {versions.tolist()} != this concopt's "
        f"RECORD_SCHEMA_VERSION ({RECORD_SCHEMA_VERSION}) -- re-record with "
        "the current recorder, or update this notebook for the new schema "
        "before trusting anything below"
    )

print(f"{len(recording)} recorded rows, {recording['elapsed_s'].iloc[-1]:,.0f} s "
      f"brake release -> touchdown")
print(recording["phase"].value_counts())


## Question 1: predicted vs actual, overall

Total time, trip fuel, and the climb / cruise / decel / level / descent /
approach split. Cut the ACTUAL flight at the same route positions
(`cum_nm`) the PREDICTED plan's own segments end at -- not at the
recorder's `phase` column, which cuts climb/cruise at the accel waypoint
(a named `.pln` fix) rather than at `conc_climb.csv`'s own top-of-climb
distance, and which has no "level" phase at all (level cruise and descent
both read as Mach < 1, post-decel -- see `inflight._flight_phase`). Cutting
by the predicted DISTANCE each segment ends at, instead of the recorder's
coarser phase label, answers the actual question: how long did it take /
how much fuel did it cost to cover the ground the model said each segment
would cover.

In [ ]:
cum_sorted_idx = np.argsort(recording["cum_nm"].to_numpy())
cum_sorted = recording["cum_nm"].to_numpy()[cum_sorted_idx]
elapsed_sorted = recording["elapsed_s"].to_numpy()[cum_sorted_idx]
weight_sorted = recording["weight_t"].to_numpy()[cum_sorted_idx]

seg_names = ["climb", "cruise", "decel", "level", "descent", "approach"]
actual_rows = []
for name, c0, c1 in zip(seg_names, predicted_cum_bounds[:-1], predicted_cum_bounds[1:]):
    t0, t1 = np.interp([c0, c1], cum_sorted, elapsed_sorted)
    w0, w1 = np.interp([c0, c1], cum_sorted, weight_sorted)
    actual_rows.append(dict(segment=name, time_s=t1 - t0, fuel_t=w0 - w1))
actual = pd.DataFrame(actual_rows).set_index("segment")

q1 = predicted.join(actual, lsuffix="_pred", rsuffix="_actual")
totals = pd.DataFrame([dict(
    time_s_pred=predicted["time_s"].sum(), time_s_actual=actual["time_s"].sum(),
    fuel_t_pred=predicted["fuel_t"].sum(), fuel_t_actual=actual["fuel_t"].sum(),
)], index=["TOTAL"])
q1 = pd.concat([q1, totals])
q1["delta_time_s"] = q1["time_s_actual"] - q1["time_s_pred"]
q1["delta_fuel_t"] = q1["fuel_t_actual"] - q1["fuel_t_pred"]
q1["delta_time_pct"] = 100.0 * q1["delta_time_s"] / q1["time_s_pred"]
q1["delta_fuel_pct"] = 100.0 * q1["delta_fuel_t"] / q1["fuel_t_pred"]

q1.round(2)


## Question 2: where the prediction drifted

`predicted_total_s` is the LIVE estimate the recorder logged at every
sample (elapsed so far + the in-flight remaining-time estimate); plotted
against `elapsed_s`, a model that's wrong in one segment shows as a step,
one that's wrong everywhere shows as a slope. The pre-flight total (one
number, from `report.csv`/section 1a) is the flat reference line the live
estimate should converge onto as touchdown approaches.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

have_pred = recording.dropna(subset=["predicted_total_s"])
ax.plot(recording["elapsed_s"] / 60.0, recording["elapsed_s"] / 60.0,
        color="grey", linestyle=":", linewidth=1, label="elapsed so far")
ax.scatter(have_pred["elapsed_s"] / 60.0, have_pred["predicted_total_s"] / 60.0,
           color=PALETTE[0], s=14, label="live predicted total (this tick)")
ax.axhline(predicted["time_s"].sum() / 60.0, color=PALETTE[1], linestyle="--",
           linewidth=1, label="pre-flight predicted total (report.csv)")
ax.axhline(recording["elapsed_s"].iloc[-1] / 60.0, color=PALETTE[2], linestyle="--",
           linewidth=1, label="actual touchdown")
ax.set_xlabel("elapsed since brake release (min)")
ax.set_ylabel("predicted total block time (min)")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Predicted total block time, as estimated at each tick")
fig.tight_layout()

n_missing = recording["predicted_total_s"].isna().sum()
print(f"predicted_total_s available on {len(have_pred)}/{len(recording)} ticks "
      f"({n_missing} missing)")
if n_missing:
    print(recording.groupby("phase")["predicted_total_s"].apply(lambda s: s.notna().sum()).rename("available_ticks"))


## Question 3: the weather chain

ERA5 (what the plan was built from) -> Active Sky (what the advisor
queried in flight) -> the sim (what the aircraft actually flew in),
along-track wind against distance flown. The ERA5-to-Active-Sky hop is
already known to be systematic at about -18.9 kt (seven `concopt verify`
runs); this is the first look at Active-Sky-to-actual. Mean and spread are
reported per hop, separately -- they may cancel end to end and hide two
real, opposite-signed errors.

`report.csv`'s `wind_kt` column only covers the climb+cruise waypoints
(the arrival's ERA5 wind is a single representative value baked into the
predicted arrival time, not logged position-by-position) -- the ERA5
series below is clipped to that span; Active Sky and the sim cover the
whole flight (`as_wind_along_kt`/`sim_wind_along_kt`, both already recorded
along the route's own leg track).

In [ ]:
era5 = report_df.dropna(subset=["wind_kt"]).sort_values("cum_nm")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(era5["cum_nm"], era5["wind_kt"], color=PALETTE[0], marker="o", markersize=3,
        label="ERA5 (predicted, report.csv)")
ax.plot(recording["cum_nm"], recording["as_wind_along_kt"], color=PALETTE[1],
        linewidth=1, label="Active Sky (queried in flight)")
ax.plot(recording["cum_nm"], recording["sim_wind_along_kt"], color=PALETTE[2],
        linewidth=1, label="sim (actually flown in)")
ax.set_xlabel("distance flown (nm)")
ax.set_ylabel("along-track wind (kt, +tailwind)")
ax.legend(fontsize=8)
ax.set_title("The weather chain: ERA5 -> Active Sky -> sim")
fig.tight_layout()

rec_by_cum = recording.sort_values("cum_nm")
in_era5_span = rec_by_cum["cum_nm"].between(era5["cum_nm"].min(), era5["cum_nm"].max())
era5_on_rec = np.interp(rec_by_cum.loc[in_era5_span, "cum_nm"], era5["cum_nm"], era5["wind_kt"])
era5_to_as_kt = rec_by_cum.loc[in_era5_span, "as_wind_along_kt"].to_numpy() - era5_on_rec
as_to_sim_kt = (recording["as_wind_along_kt"] - recording["sim_wind_along_kt"]).dropna().to_numpy()

print(f"ERA5 -> Active Sky:  mean {np.nanmean(era5_to_as_kt):+.2f} kt, "
      f"std {np.nanstd(era5_to_as_kt):.2f} kt  (n={np.isfinite(era5_to_as_kt).sum()}, "
      f"climb+cruise span only)")
print(f"Active Sky -> sim:   mean {np.nanmean(as_to_sim_kt):+.2f} kt, "
      f"std {np.nanstd(as_to_sim_kt):.2f} kt  (n={len(as_to_sim_kt)}, whole flight)")


## Question 4: the advice

Every CLIMB/DESCEND recommendation the advisor gave (`rec_action`,
`rec_fl`, `rec_gain_kt`/`rec_gain_s` -- HOLD ticks carry no actionable
advice and are excluded), and whether the flown level actually moved to
within 500 ft of `rec_fl` inside the next couple of minutes. Summed
`rec_gain_s` by followed/not-followed is an ESTIMATE of what following
every recommendation would have been worth, not a measurement -- the
counterfactual (the OTHER path) was never actually flown, and the chart
says so directly rather than only in a footnote.

In [ ]:
LOOKAHEAD_WINDOW_S = 150.0  # "a couple of minutes"
FL_TOLERANCE = 5.0  # +-500 ft

rec_sorted = recording.sort_values("elapsed_s").reset_index(drop=True)
els = rec_sorted["elapsed_s"].to_numpy()
fl = rec_sorted["level_fl"].to_numpy()

advice = rec_sorted[rec_sorted["rec_action"].isin(["CLIMB", "DESCEND"])].copy()
followed = []
for _, row in advice.iterrows():
    window = (els >= row["elapsed_s"]) & (els <= row["elapsed_s"] + LOOKAHEAD_WINDOW_S)
    followed.append(bool(np.any(np.abs(fl[window] - row["rec_fl"]) <= FL_TOLERANCE)))
advice["followed"] = followed

print(f"{len(advice)}/{len(rec_sorted)} ticks carried a CLIMB/DESCEND recommendation")
if len(advice):
    print(advice.groupby("followed")[["rec_gain_s", "rec_gain_kt"]].agg(["count", "sum", "mean"]).round(1))

fig, ax = plt.subplots(figsize=(6, 4.5))
if len(advice):
    sums = advice.groupby("followed")["rec_gain_s"].sum().reindex([True, False], fill_value=0.0)
    bars = ax.bar(["followed", "not followed"], sums.values,
                   color=[PALETTE[2], PALETTE[3]])
    ax.bar_label(bars, fmt="%.0f s")
ax.set_ylabel("summed predicted gain, rec_gain_s")
ax.set_title("ESTIMATE, not a measurement -- the counterfactual was never flown")
fig.tight_layout()


## Question 5: the assumed constants

`arrival.APPROACH_NM`/`APPROACH_MIN`/`APPROACH_FUEL_T` are ASSUMPTIONS, not
table values (see that module's own comment) -- this is what one flight
says they should actually be. Measured from 1,500 ft AGL to touchdown, the
same cut `inflight._flight_phase` uses for `phase == "approach"`. The
recorder only samples every `LOW_ALT_INTERVAL_S` near the ground (5 s
wall-clock), so the exact instant `agl_ft` crosses 1,500 ft is interpolated
between the two straddling samples rather than snapped to the nearest
one -- snapping to a row was off by close to a full tick against the
assumed constants; interpolating recovers them much more closely.

In [ ]:
near_ground = recording.sort_values("elapsed_s")
agl = near_ground["agl_ft"].to_numpy()
els5 = near_ground["elapsed_s"].to_numpy()

crossing_idx = np.flatnonzero((agl[:-1] >= 1500.0) & (agl[1:] < 1500.0))[-1]
frac = (1500.0 - agl[crossing_idx]) / (agl[crossing_idx + 1] - agl[crossing_idx])
entry_s = float(els5[crossing_idx] + frac * (els5[crossing_idx + 1] - els5[crossing_idx]))
entry_cum_nm = float(np.interp(entry_s, els5, near_ground["cum_nm"].to_numpy()))
entry_weight_t = float(np.interp(entry_s, els5, near_ground["weight_t"].to_numpy()))

touchdown_cum_nm = float(recording["cum_nm"].iloc[-1])
touchdown_s = float(recording["elapsed_s"].iloc[-1])
touchdown_weight_t = float(recording["weight_t"].iloc[-1])

measured_nm = touchdown_cum_nm - entry_cum_nm
measured_min = (touchdown_s - entry_s) / 60.0
measured_fuel_t = entry_weight_t - touchdown_weight_t

pd.DataFrame([
    dict(quantity="distance, nm", assumed=arrival.APPROACH_NM, measured=round(measured_nm, 2)),
    dict(quantity="time, min", assumed=arrival.APPROACH_MIN, measured=round(measured_min, 2)),
    dict(quantity="fuel, t", assumed=arrival.APPROACH_FUEL_T, measured=round(measured_fuel_t, 3)),
]).set_index("quantity")
